# **CNN Architecture Implementation**

**Laboratory Activity 6:** *CNN Implementation*

___


## **Objective**

Convert the given CNN architecture diagram into a working PyTorch model and verify that the tensor shapes are consistent throughout the network.


## **Laboratory Problem**
**Instruction:** Convert the following CNN architecture diagram into a PyTorch CNN Architecture.

![CNN architecture diagram](figures/cnn_diagram.jpg)


## **Implementation**

### **Determine the Output Shapes**

The spatial output size of a convolution or pooling layer is:

$$
\text{Output Size}
=
\left\lfloor
\frac{W-F+2P}{S}
\right\rfloor+1
$$

Following the values shown in the diagram:

- **Conv1:** $28 \rightarrow 28$
- **MaxPool1:** $28 \rightarrow 29$
- **Conv2:** $29 \rightarrow 29$
- **Conv3:** $29 \rightarrow 29$
- **Conv4:** $29 \rightarrow 29$
- **MaxPool2:** $29 \rightarrow 28$

Therefore, the final convolutional output has shape:

$$
(256,28,28)
$$

and the flattened input to the first fully connected layer is:

$$
256\times28\times28=200704
$$

The diagram does not specify the number of output classes. Since the input format corresponds to the MNIST setting used in the lecture, the final layer is configured with 10 outputs representing digits 0–9.

In [1]:
def calc_out(w, f, s, p):
    return (w - f + 2 * p) // s + 1

# Convolution and pooling output sizes
out_conv1 = calc_out(28, 3, 1, 1)
out_pool1 = calc_out(out_conv1, 2, 1, 1)

out_conv2 = calc_out(out_pool1, 3, 1, 1)
out_conv3 = calc_out(out_conv2, 3, 1, 1)
out_conv4 = calc_out(out_conv3, 3, 1, 1)
out_pool2 = calc_out(out_conv4, 2, 1, 0)

flatten_size = 256 * out_pool2 * out_pool2

print(f"Conv1 output:    {out_conv1}x{out_conv1}")
print(f"MaxPool1 output: {out_pool1}x{out_pool1}")
print(f"Conv2 output:    {out_conv2}x{out_conv2}")
print(f"Conv3 output:    {out_conv3}x{out_conv3}")
print(f"Conv4 output:    {out_conv4}x{out_conv4}")
print(f"MaxPool2 output: {out_pool2}x{out_pool2}")
print(f"Flatten size:    {flatten_size}")

Conv1 output:    28x28
MaxPool1 output: 29x29
Conv2 output:    29x29
Conv3 output:    29x29
Conv4 output:    29x29
MaxPool2 output: 28x28
Flatten size:    200704


### **Define the CNN Architecture**

The network is implemented directly from the diagram using convolutional layers, ReLU activations, max-pooling, dropout, fully connected layers, and Softmax.

In [2]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
0.26.0+cu128
True


In [3]:
import torch.nn as nn
import torch.nn.functional as F

In [4]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        self.pool1 = nn.MaxPool2d(
            kernel_size=(2, 2),
            stride=1,
            padding=1
        )

        self.conv2 = nn.Conv2d(32, 64, (3, 3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, (3, 3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, (3, 3), stride=1, padding=1)

        self.pool2 = nn.MaxPool2d(
            kernel_size=(2, 2),
            stride=1,
            padding=0
        )

        # Dropout
        self.dropout = nn.Dropout(p=0.2)

        # Fully connected layers
        self.fcn1 = nn.Linear(256 * 28 * 28, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)

        return x

### **Instantiate the Model**

In [5]:
model = CNN(num_classes=10)
model

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=1, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=1, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=200704, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

### **Verify the Architecture**

A dummy batch of 32 grayscale $28\times28$ images is passed through the network to confirm that the architecture runs without shape errors and produces 10 class probabilities for each sample.

In [6]:
dummy_input = torch.randn(32, 1, 28, 28)

model.eval()
with torch.no_grad():
    output = model(dummy_input)

print("Input shape: ", dummy_input.shape)
print("Output shape:", output.shape)
print(
    "Softmax check:",
    torch.allclose(
        output.sum(dim=1),
        torch.ones(output.size(0)),
        atol=1e-5
    )
)

Input shape:  torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 10])
Softmax check: True


## **Results and Discussion**

The implemented CNN successfully follows the architecture shown in the diagram. The first convolution preserves the $28\times28$ input size, while the first max-pooling layer increases the spatial size to $29\times29$ because the diagram specifies a kernel size of $2$, stride of $1$, and padding of $1$. The following convolutional layers preserve this size while increasing the number of feature channels from $32$ to $256$. The second max-pooling layer then returns the spatial size to $28\times28$, producing $200704$ values after flattening. A dummy batch with shape $(32,1,28,28)$ passes through the model successfully and produces an output with shape $(32,10)$, confirming that the dimensions are compatible throughout the network.

## **Conclusion**

This laboratory demonstrated how a CNN architecture diagram can be translated into a PyTorch `nn.Module` by carefully following the specified channels, kernel sizes, strides, padding, pooling, dropout, and fully connected layers. Calculating the output dimensions before defining the fully connected layers was necessary to obtain the correct flatten size. The successful dummy forward pass confirmed that the implemented architecture is dimensionally consistent with the given design.